In [ ]:
import gurobipy as gp
from gurobipy import GRB, Model
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# Helper function to load instances from Excel file
def load_instance(filename):
    # Load instance from Excel file (one workbook with five sheets).
    
    parameters_df = pd.read_excel(filename, sheet_name='parameters')
    relays_df = pd.read_excel(filename, sheet_name='relays')
    home_bases_df = pd.read_excel(filename, sheet_name='home_bases')
    shipments_df = pd.read_excel(filename, sheet_name='shipments')
    distance_matrix_df = pd.read_excel(filename, sheet_name='distance_matrix')

    # Convert parameters sheet to a flat dict with names and values
    params = dict(zip(parameters_df['parameter'], parameters_df['value']))

    # Force integer types for parameters that index sets
    for k in ['n_shipments', 'n_drivers', 'n_relays', 'T_periods',
              'horizon_hours', 'period_minutes', 'handoff_min_periods',
              'duty_bound_periods', 'cap_per_relay_per_period',
              'origin_start_window_periods', 'periods_per_day',
              'shift_open_offset', 'shift_close_offset',
              'relay_dep_buffer_at_end', 'dest_end_buffer', 'seed']:
        if k in params:
            params[k] = int(params[k])

    return params, relays_df, home_bases_df, shipments_df, distance_matrix_df


print("Helper functions loaded successfully!")


# Helper functions for pre-processing


def build_node_id(kind, key, suffix=None):
    
    # Generate a node identifier.

    # ('home_base', 'B1') -> 'B_B1'
    # ('origin', 'F1')    -> 'O_F1'
    # ('destination', 'F1') -> 'D_F1'
    # ('relay', 'R1', 'arrival') -> 'R1_a'
    # ('relay', 'R1', 'departure') -> 'R1_d'
    
    if kind == 'home_base':
        return f'B_{key}'
    if kind == 'origin':
        return f'O_{key}'
    if kind == 'destination':
        return f'D_{key}'
    if kind == 'relay':
        suf = 'a' if suffix == 'arrival' else 'd'
        return f'{key}_{suf}'
    raise ValueError(kind)


def build_distance_dict(distance_matrix_df, relays_df, home_bases_df, shipments_df):

    # Build a dict mapping (loc1_id, loc2_id) -> (distance_km, travel_time_periods).

    # The distance_matrix sheet uses node IDs like 'relay_R1_arrival', 'B1', 'origin_F1', 'destination_F1' -> re-define node IDs
    

    rename_map = {}
    for _, r in relays_df.iterrows():
        rename_map[f'relay_{r.relay_id}_arrival'] = build_node_id('relay', r.relay_id, 'arrival')
        rename_map[f'relay_{r.relay_id}_departure'] = build_node_id('relay', r.relay_id, 'departure')
    for _, b in home_bases_df.iterrows():
        rename_map[b.home_base_id] = build_node_id('home_base', b.home_base_id)
    for _, s in shipments_df.iterrows():
        rename_map[f'origin_{s.shipment_id}'] = build_node_id('origin', s.shipment_id)
        rename_map[f'destination_{s.shipment_id}'] = build_node_id('destination', s.shipment_id)

    dist = {}
    for _, row in distance_matrix_df.iterrows():
        a = rename_map.get(row['from_node'])
        b = rename_map.get(row['to_node'])
        if a is None or b is None:
            continue
        dist[(a, b)] = (float(row['distance_km']), int(row['travel_time_periods']))
    return dist


# def physical_location_of(node_id, relays_df):
#     """Return the underlying physical location id of a node, treating relay
#     arrival and departure layers as the same physical relay for distance
#     purposes. E.g. 'R3_a' and 'R3_d' both share the lookup of 'R3_a'."""
#     return node_id


# Main solver

def solve_relay_model(params, relays_df, home_bases_df, shipments_df,
                     distance_matrix_df,
                     time_limit_seconds=600,
                     mip_gap=0.01,
                     verbose=True):


    # Index sets

    F = list(shipments_df['shipment_id'])                    # shipments
    D = list(home_bases_df['driver_id'])                     # drivers
    I = list(relays_df['relay_id'])                          # relay points
    T = int(params['T_periods'])                             # planning horizon
    Tset = list(range(T + 1))                                # 0..T intervals


    n_F, n_D, n_I = len(F), len(D), len(I)
    if verbose:
        print(f"\n=== Sets ===  |F|={n_F}  |D|={n_D}  |I|={n_I}  T={T}")



    # Index lookups
 
    ship = shipments_df.set_index('shipment_id')
    drv = home_bases_df.set_index('driver_id')
    rly = relays_df.set_index('relay_id')

    # Data of each shipment

    O_of = {f: build_node_id('origin', f) for f in F}
    D_of = {f: build_node_id('destination', f) for f in F}
    tO_under = {f: int(ship.at[f, 't_O_under']) for f in F}
    tO_bar = {f: int(ship.at[f, 't_O_bar']) for f in F}
    tD_under = {f: int(ship.at[f, 't_D_under']) for f in F}
    tD_bar = {f: int(ship.at[f, 't_D_bar']) for f in F}
    Dhat = {f: float(ship.at[f, 'direct_km']) for f in F}

    # Data of each driver
    b_of = {d: build_node_id('home_base', drv.at[d, 'home_base_id']) for d in D}
    cd = {d: float(drv.at[d, 'cost_deployment_eur']) for d in D}

    # Data of each RP

    Ra = {i: build_node_id('relay', i, 'arrival') for i in I}
    Rd = {i: build_node_id('relay', i, 'departure') for i in I}
    tRd_under = {i: int(rly.at[i, 't_id_under']) for i in I}
    tRd_bar = {i: int(rly.at[i, 't_id_bar']) for i in I}
    tau_min = {i: int(rly.at[i, 'handoff_min_periods']) for i in I}
    Cap = {i: int(rly.at[i, 'capacity_per_period']) for i in I}

    # Other parameters for objective & constraints

    c_r = float(params['cost_loaded_eur_per_km'])
    c_e = float(params['cost_empty_eur_per_km'])
    c_w = float(params['cost_waiting_eur_per_period'])
    M_unserved = float(params['penalty_unserved_eur'])
    H = int(params['duty_bound_periods'])
    omega = float(params['omega_circuity'])

 
 
    # Distance & travel-time lookup


    dist = build_distance_dict(distance_matrix_df, relays_df, home_bases_df, shipments_df)

    def D_arc(u, v):
        # Distance 

        if (u, v) in dist:
            return dist[(u, v)][0]
        if (v, u) in dist:
            return dist[(v, u)][0]
        return 0.0

    def L_arc(u, v):
        # Travel time

        if (u, v) in dist:
            return dist[(u, v)][1]
        if (v, u) in dist:
            return dist[(v, u)][1]
        return 0

    
    
    # Build arc sets with each arc as a tuple (v1, v2, t1, t2) 


    A_loc_in = []     # (O_f, R_a, t, t+l)
    A_loc_out = []    # (R_d, D_f, t, t+l)
    A_rel = []        # (R_d, R'_a, t, t+l)
    A_dir = []        # (O_f, D_f, t, t+l)
    A_trans = []      # (R_a, R_d, t, t+tau)
    A_dead_DR = []    # (D_f, R_a, t, t+l)
    A_dead_DO = []    # (D_f, O_g, t, t+l)
    A_dead_RO = []    # (R_d, O_f, t, t+l)
    A_acc = {d: [] for d in D}     # driver only
    A_egr = {d: [] for d in D}     # driver only
    A_wait_f = {f: [] for f in F}  # trailer only
    A_wait_d = []                  # driver only
    A_HB = {d: [] for d in D}      # driver waiting only
    A_dum_d = {d: (b_of[d], b_of[d], 0, T) for d in D}   # driver dummy arc


    # Local inbound (origin -> relay arrival) 
    for f in F:
        for i in I:
            ell = L_arc(O_of[f], Ra[i])
            if ell > H or ell == 0:
                continue
            tmin = tO_under[f]
            tmax = min(tO_bar[f], tRd_bar[i] - tau_min[i] - ell)
            for t in range(tmin, tmax + 1):
                if 0 <= t and t + ell <= T:
                    A_loc_in.append((O_of[f], Ra[i], t, t + ell))

    # Local outbound (relay departure -> destination)
    for f in F:
        for i in I:
            ell = L_arc(Rd[i], D_of[f])
            if ell > H or ell == 0:
                continue
            tmin = tRd_under[i]
            tmax = tRd_bar[i]
            for t in range(tmin, tmax + 1):
                if t + ell <= T:
                    A_loc_out.append((Rd[i], D_of[f], t, t + ell))

    # Relay arcs (relay departure -> another relay arrival)
    for i in I:
        for j in I:
            if i == j:
                continue
            ell = L_arc(Rd[i], Ra[j])
            if ell > H or ell == 0:
                continue
            tmin = tRd_under[i]
            tmax = min(tRd_bar[i], tRd_bar[j] - tau_min[j] - ell)
            for t in range(tmin, tmax + 1):
                if t + ell <= T:
                    A_rel.append((Rd[i], Ra[j], t, t + ell))

    # Direct arcs (origin -> destination, only if l <= H) 
    for f in F:
        ell = L_arc(O_of[f], D_of[f])
        if ell == 0 or ell > H:
            continue
        tmin = tO_under[f]
        tmax = tO_bar[f]
        for t in range(tmin, tmax + 1):
            if t + ell <= T:
                A_dir.append((O_of[f], D_of[f], t, t + ell))

    # Transition arcs (relay arrival -> relay departure)
    for i in I:
        ell = tau_min[i]
        for t in range(T - ell + 1):
            if t + ell <= tRd_bar[i]:
                A_trans.append((Ra[i], Rd[i], t, t + ell))

    # Deadhead destination -> relay arrival 
    for f in F:
        for i in I:
            ell = L_arc(D_of[f], Ra[i])
            if ell > H or ell == 0:
                continue
            tmin = tD_under[f]
            tmax = min(tD_bar[f], tRd_bar[i] - tau_min[i] - ell)
            for t in range(tmin, tmax + 1):
                if t + ell <= T:
                    A_dead_DR.append((D_of[f], Ra[i], t, t + ell))

    # Deadhead destination -> origin 
    for f in F:
        for g in F:
            if f == g:
                continue
            ell = L_arc(D_of[f], O_of[g])
            if ell > H or ell == 0:
                continue
            tmin = tD_under[f]
            tmax = min(tD_bar[f], tO_bar[g] - ell)
            for t in range(tmin, tmax + 1):
                if t + ell <= T:
                    A_dead_DO.append((D_of[f], O_of[g], t, t + ell))

    # Deadhead relay departure -> origin 
    for i in I:
        for f in F:
            ell = L_arc(Rd[i], O_of[f])
            if ell > H or ell == 0:
                continue
            tmin = tRd_under[i]
            tmax = min(tRd_bar[i], tO_bar[f] - ell)
            for t in range(tmin, tmax + 1):
                if t + ell <= T:
                    A_dead_RO.append((Rd[i], O_of[f], t, t + ell))

    # Access arcs (home base -> first served node), for each driver
    for d in D:
        b = b_of[d]
        # to origin node
        for f in F:
            ell = L_arc(b, O_of[f])
            if ell > H or ell == 0:
                continue
            for t in range(T):
                if t + ell <= tO_bar[f] and t + ell <= T:
                    A_acc[d].append((b, O_of[f], t, t + ell))
        # to relay arrival
        for i in I:
            ell = L_arc(b, Ra[i])
            if ell > H or ell == 0:
                continue
            for t in range(T):
                if t + ell <= tRd_bar[i] - tau_min[i] and t + ell <= T:
                    A_acc[d].append((b, Ra[i], t, t + ell))

    # Egress arcs (last served node -> home base), for each driver
    for d in D:
        b = b_of[d]
        # from destination node
        for f in F:
            ell = L_arc(D_of[f], b)
            if ell > H or ell == 0:
                continue
            tmin = tD_under[f]
            tmax = tD_bar[f]
            for t in range(tmin, tmax + 1):
                if t + ell <= T:
                    A_egr[d].append((D_of[f], b, t, t + ell))
        # from relay departure
        for i in I:
            ell = L_arc(Rd[i], b)
            if ell > H or ell == 0:
                continue
            tmin = tRd_under[i]
            tmax = tRd_bar[i]
            for t in range(tmin, tmax + 1):
                if t + ell <= T:
                    A_egr[d].append((Rd[i], b, t, t + ell))

    # Trailer waiting arcs 
    for f in F:
        # at origin (before the latest start TW)
        for t in range(0, tO_bar[f]):
            A_wait_f[f].append((O_of[f], O_of[f], t, t + 1))
        # at relay departure (before the latest start TW)
        for i in I:
            for t in range(0, tRd_bar[i]):
                A_wait_f[f].append((Rd[i], Rd[i], t, t + 1))
        # at destination (before the end of the planning horizon)
        for t in range(0, T):
            A_wait_f[f].append((D_of[f], D_of[f], t, t + 1))

    # Driver waiting arcs at relay departure
    for i in I:
        for t in range(0, tRd_bar[i]):
            A_wait_d.append((Rd[i], Rd[i], t, t + 1))

    # Driver waiting arcs at their home base (before access & after egress)
    for d in D:
        for t in range(0, T):
            A_HB[d].append((b_of[d], b_of[d], t, t + 1))

    # Aggregate sets
    A_loc = A_loc_in + A_loc_out
    A_loaded = A_loc + A_rel + A_dir
    A_dead = A_dead_DR + A_dead_DO + A_dead_RO

    if verbose:
        print(f"=== Arcs ===")
        print(f"  loc_in:    {len(A_loc_in):>3d}")
        print(f"  loc_out:   {len(A_loc_out):>3d}")
        print(f"  relay:     {len(A_rel):>3d}")
        print(f"  direct:    {len(A_dir):>3d}")
        print(f"  trans:     {len(A_trans):>3d}")
        print(f"  dead_DR:   {len(A_dead_DR):>3d}")
        print(f"  dead_DO:   {len(A_dead_DO):>3d}")
        print(f"  dead_RO:   {len(A_dead_RO):>3d}")
        print(f"  access:    {sum(len(v) for v in A_acc.values()):>3d}")
        print(f"  egress:    {sum(len(v) for v in A_egr.values()):>3d}")
        print(f"  wait_f:    {sum(len(v) for v in A_wait_f.values()):>3d}")
        print(f"  wait_d:    {len(A_wait_d):>3d}")
        print(f"  HB:        {sum(len(v) for v in A_HB.values()):>3d}")
        print(f"  dum_d:     {len(A_dum_d)}")

   
    # Successor & predecessor lookups for each trailer and driver
    
    
    # An admissible arc set for trailer:
    A_f_per = {f: [] for f in F}
    for f in F:
        # local inbound and outbound only for this shipment
        for a in A_loc_in:
            if a[0] == O_of[f]:
                A_f_per[f].append(a)
        for a in A_loc_out:
            if a[1] == D_of[f]:
                A_f_per[f].append(a)
        # relay, direct only for this shipment if feasible, transition, waiting
        for a in A_rel:
            A_f_per[f].append(a)
        for a in A_dir:
            if a[0] == O_of[f] and a[1] == D_of[f]:
                A_f_per[f].append(a)
        for a in A_trans:
            A_f_per[f].append(a)
        for a in A_wait_f[f]:
            A_f_per[f].append(a)

    # An admissible arc set for driver
    A_d_per = {d: [] for d in D}
    for d in D:
        # access & egress only for this driver
        A_d_per[d].extend(A_acc[d])
        A_d_per[d].extend(A_egr[d])
        # loaded movement arcs (with trailers)
        A_d_per[d].extend(A_loc_in)
        A_d_per[d].extend(A_loc_out)
        A_d_per[d].extend(A_rel)
        A_d_per[d].extend(A_dir)
        A_d_per[d].extend(A_trans)
        # deadhead (driver only)
        A_d_per[d].extend(A_dead_DR)
        A_d_per[d].extend(A_dead_DO)
        A_d_per[d].extend(A_dead_RO)
        # waiting
        A_d_per[d].extend(A_wait_d)
        A_d_per[d].extend(A_HB[d])
        A_d_per[d].append(A_dum_d[d])

    # successor & predecessor by (node, time)
    def build_in_out(arcs):
        out = {}; inn = {}
        for a in arcs:
            v1, v2, t1, t2 = a
            out.setdefault((v1, t1), []).append(a)
            inn.setdefault((v2, t2), []).append(a)
        return out, inn

    out_f = {}; inn_f = {}
    for f in F:
        out_f[f], inn_f[f] = build_in_out(A_f_per[f])
    out_d = {}; inn_d = {}
    for d in D:
        out_d[d], inn_d[d] = build_in_out(A_d_per[d])

   
   
    # Build the model

    m = Model("Relay_Trucking")

    # Trailer flow: x^f, a binary in set of admissible arcs for trailers
    x_f = {}
    for f in F:
        for a in A_f_per[f]:
            x_f[(f, a)] = m.addVar(vtype=GRB.BINARY, name=f"x[{f},{a}]")

    # Driver flow: y^d, a binary in set of admissible arcs for drivers
    y_d = {}
    for d in D:
        for a in A_d_per[d]:
            y_d[(d, a)] = m.addVar(vtype=GRB.BINARY, name=f"y[{d},{a}]")

    # Unserved shipment
    u = {f: m.addVar(vtype=GRB.BINARY, name=f"u[{f}]") for f in F}

    m.update()

 
    # Objective function
    
    
    # (1) deployment cost (per driver c^d_d on access arcs)
    term_deploy = gp.quicksum(
        cd[d] * y_d[(d, a)]
        for d in D for a in A_acc[d]
    )

    # (2) loaded routing cost on x^f over loaded arcs (D_a * x)

    A_loaded_set = set(A_loaded)
    term_loaded = gp.quicksum(
        c_r * D_arc(a[0], a[1]) * x_f[(f, a)]
        for f in F for a in A_f_per[f] if a in A_loaded_set
    )

    # (3) empty miles cost on deadhead arcs with only drivers
    A_empty_per_d = {d: list(A_dead) for d in D}
    term_empty_solo = gp.quicksum(
        c_e * D_arc(a[0], a[1]) * y_d[(d, a)]
        for d in D for a in A_empty_per_d[d]
    )

    # (4) empty miles cost on relay arcs with only drivers, no trailers: c_e * D * (sum_d y - sum_f x) 

    term_empty_rel = gp.quicksum(
        c_e * D_arc(a[0], a[1]) * (
            gp.quicksum(y_d[(d, a)] for d in D)
            - gp.quicksum(x_f[(f, a)] for f in F)
        )
        for a in A_rel
    )

    # (6) trailer waiting cost, only charged at relay departure
    Rd_set = set(Rd.values())
    term_wait = gp.quicksum(
        c_w * (a[3] - a[2]) * x_f[(f, a)]
        for f in F for a in A_wait_f[f]
        if a[0] in Rd_set and a[1] in Rd_set
    )

    # (6) penalties for unserved shipments

    term_unserved = gp.quicksum(M_unserved * u[f] for f in F)

    m.setObjective(
        term_deploy + term_loaded + term_empty_solo + term_empty_rel
        + term_wait + term_unserved,
        GRB.MINIMIZE
    )

 
    # Constraints
  

    # Driver flow conservation: per driver, per time-space node
    nodes_per_driver = {}
    for d in D:
        nodes = set()
        for a in A_d_per[d]:
            v1, v2, t1, t2 = a
            nodes.add((v1, t1))
            nodes.add((v2, t2))
        nodes_per_driver[d] = nodes

    for d in D:
        b = b_of[d]
        for (v, t) in nodes_per_driver[d]:
            outflow = gp.quicksum(y_d[(d, a)] for a in out_d[d].get((v, t), []))
            inflow = gp.quicksum(y_d[(d, a)] for a in inn_d[d].get((v, t), []))
            if v == b and t == 0:
                rhs = 1
            elif v == b and t == T:
                rhs = -1
            else:
                rhs = 0
            m.addConstr(outflow - inflow == rhs,
                        name=f"DrvFlow[{d},{v},{t}]")

    # Duty time limit: sum t2*y_egr - sum t1*y_acc <= H
    for d in D:
        m.addConstr(
            gp.quicksum(a[3] * y_d[(d, a)] for a in A_egr[d])
            - gp.quicksum(a[2] * y_d[(d, a)] for a in A_acc[d])
            <= H,
            name=f"Duty[{d}]"
        )

    # Trailer flow conservation, with unserved indicator on source/sink node
    nodes_per_ship = {}
    for f in F:
        nodes = set()
        for a in A_f_per[f]:
            v1, v2, t1, t2 = a
            nodes.add((v1, t1))
            nodes.add((v2, t2))
        nodes_per_ship[f] = nodes

    for f in F:
        for (v, t) in nodes_per_ship[f]:
            outflow = gp.quicksum(x_f[(f, a)] for a in out_f[f].get((v, t), []))
            inflow = gp.quicksum(x_f[(f, a)] for a in inn_f[f].get((v, t), []))
            if v == O_of[f] and t == 0:
                m.addConstr(outflow - inflow == 1 - u[f],
                            name=f"FrtFlow_src[{f}]")
            elif v == D_of[f] and t == T:
                m.addConstr(outflow - inflow == -(1 - u[f]),
                            name=f"FrtFlow_snk[{f}]")
            else:
                m.addConstr(outflow - inflow == 0,
                            name=f"FrtFlow[{f},{v},{t}]")

    # Coupling: equality on local + direct + transition arcs

    A_couple_eq_set = set(A_loc) | set(A_dir) | set(A_trans)
    for a in A_couple_eq_set:
        m.addConstr(
            gp.quicksum(x_f[(f, a)] for f in F if (f, a) in x_f)
            == gp.quicksum(y_d[(d, a)] for d in D if (d, a) in y_d),
            name=f"CoupleEq[{a}]"
        )

    # Coupling: inequality on relay arcs (drivers can travel without trailers)
    for a in A_rel:
        m.addConstr(
            gp.quicksum(x_f[(f, a)] for f in F if (f, a) in x_f)
            <= gp.quicksum(y_d[(d, a)] for d in D if (d, a) in y_d),
            name=f"CoupleIneq[{a}]"
        )

    # Capacity at relay point: total trailers in arrival layer <= Cap

    # Group inbound trailer arcs by (relay_arrival, head_time)

    arrivals_at_relay = {(i, t): [] for i in I for t in Tset}
    for a in A_loc_in + A_rel:
        v2, t2 = a[1], a[3]
        # find which relay this arrival belongs to
        for i in I:
            if v2 == Ra[i]:
                arrivals_at_relay[(i, t2)].append(a)
                break

    for i in I:
        for t in Tset:
            arcs_here = arrivals_at_relay[(i, t)]
            if not arcs_here:
                continue
            m.addConstr(
                gp.quicksum(x_f[(f, a)] for f in F for a in arcs_here if (f, a) in x_f)
                <= Cap[i],
                name=f"Cap[{i},{t}]"
            )

    # Circuity: total loaded distance (trailers are carried) <= (1+omega) * Dhat * (1 - u_f)
    for f in F:
        m.addConstr(
            gp.quicksum(D_arc(a[0], a[1]) * x_f[(f, a)]
                        for a in A_f_per[f] if a in A_loaded_set)
            <= (1 + omega) * Dhat[f] * (1 - u[f]),
            name=f"Circuity[{f}]"
        )

    m.update()

    if verbose:
        print(f"Model size")
        print(f"  variables:   {m.NumVars}")
        print(f"  constraints: {m.NumConstrs}")

    
    
    # Solve
 
    m.Params.TimeLimit = time_limit_seconds
    m.Params.MIPGap = mip_gap
    m.optimize()

    vars_out = {
        'x_f': x_f,
        'y_d': y_d,
        'u': u,
        'F': F, 'D': D, 'I': I, 'T': T, 'Tset': Tset,
        'A_loc_in': A_loc_in, 'A_loc_out': A_loc_out,
        'A_rel': A_rel, 'A_dir': A_dir, 'A_trans': A_trans,
        'A_dead_DR': A_dead_DR, 'A_dead_DO': A_dead_DO,
        'A_dead_RO': A_dead_RO,
        'A_acc': A_acc, 'A_egr': A_egr,
        'A_wait_f': A_wait_f, 'A_wait_d': A_wait_d,
        'A_HB': A_HB, 'A_dum_d': A_dum_d,
        'A_loaded': A_loaded, 'A_dead': A_dead,
        'O_of': O_of, 'D_of': D_of,
        'Ra': Ra, 'Rd': Rd, 'b_of': b_of,
        'cd': cd, 'Dhat': Dhat,
        'D_arc': D_arc, 'L_arc': L_arc,
        'c_r': c_r, 'c_e': c_e, 'c_w': c_w, 'M_unserved': M_unserved,
        'H': H, 'omega': omega,
    }
    return m, vars_out


# Output / post-processing


def print_solution(m, vars_out, instance_name=''):
    # Print objective breakdown and a quick summary of the solution

    if m.Status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL, GRB.TIME_LIMIT):
        print(f"Solver ended with status {m.Status} - no usable solution.")
        return

    F = vars_out['F']
    D = vars_out['D']
    I = vars_out['I']
    x_f = vars_out['x_f']
    y_d = vars_out['y_d']
    u = vars_out['u']
    A_acc = vars_out['A_acc']
    A_egr = vars_out['A_egr']
    A_dead = vars_out['A_dead']
    A_loaded = vars_out['A_loaded']
    A_rel = vars_out['A_rel']
    A_wait_f = vars_out['A_wait_f']
    Rd = vars_out['Rd']
    cd = vars_out['cd']
    c_r = vars_out['c_r']
    c_e = vars_out['c_e']
    c_w = vars_out['c_w']
    M_unserved = vars_out['M_unserved']
    D_arc = vars_out['D_arc']

    # Driver deployment
    deployed = [d for d in D if any(y_d[(d, a)].X > 0.5 for a in A_acc[d])]
    deploy_cost = sum(cd[d] for d in deployed)

    # Loaded movements
    A_loaded_set = set(A_loaded)
    loaded_km = sum(D_arc(a[0], a[1]) * x_f[(f, a)].X
                    for (f, a) in x_f if a in A_loaded_set)
    loaded_cost = c_r * loaded_km

    # Empty miles on driver deadhead arcs
    empty_solo_km = 0.0
    for d in D:
        for a in list(A_dead):
            if (d, a) in y_d and y_d[(d, a)].X > 0.5:
                empty_solo_km += D_arc(a[0], a[1])
    empty_solo_cost = c_e * empty_solo_km

    # Empty miles on relay arcs
    empty_rel_km = 0.0
    for a in A_rel:
        sum_y = sum(y_d[(d, a)].X for d in D if (d, a) in y_d)
        sum_x = sum(x_f[(f, a)].X for f in F if (f, a) in x_f)
        empty_rel_km += D_arc(a[0], a[1]) * max(0.0, sum_y - sum_x)
    empty_rel_cost = c_e * empty_rel_km

    # Waiting cost of trailers at RP
    Rd_set = set(Rd.values())
    wait_periods = 0.0
    for f in F:
        for a in A_wait_f[f]:
            if a[0] in Rd_set and a[1] in Rd_set:
                if x_f[(f, a)].X > 0.5:
                    wait_periods += (a[3] - a[2])
    wait_cost = c_w * wait_periods

    # Unserved shipments
    unserved = [f for f in F if u[f].X > 0.5]
    unserved_cost = M_unserved * len(unserved)

    total = deploy_cost + loaded_cost + empty_solo_cost + empty_rel_cost + wait_cost + unserved_cost

    print(f"\n Solution{(' for ' + instance_name) if instance_name else ''}")
    print(f"Solver status:    {m.Status}  (gap {m.MIPGap*100:.2f}%, time {m.Runtime:.1f}s)")
    print(f"Drivers deployed: {len(deployed)}/{len(D)}  -> {deployed}")
    print(f"Shipments served: {len(F) - len(unserved)}/{len(F)}", end='')
    if unserved:
        print(f"  (unserved: {unserved})")
    else:
        print()
    print()
    print(f"Objective breakdown:")
    print(f"  Deployment cost:     {deploy_cost:>12,.2f} EUR  ({len(deployed)} duties)")
    print(f"  Loaded routing cost:      {loaded_cost:>12,.2f} EUR  ({loaded_km:>10,.0f} km)")
    print(f"  Empty on deadheads:{empty_solo_cost:>12,.2f} EUR  ({empty_solo_km:>10,.0f} km)")
    print(f"  Empty on relay arcs: {empty_rel_cost:>12,.2f} EUR  ({empty_rel_km:>10,.0f} km)")
    print(f"  Waiting at relays:   {wait_cost:>12,.2f} EUR  ({wait_periods:>10,.0f} periods)")
    print(f"  Unserved penalty:    {unserved_cost:>12,.2f} EUR  ({len(unserved)} shipments)")
    print(f"  Total:               {total:>12,.2f} EUR")
    print(f"  Gurobi ObjVal:       {m.ObjVal:>12,.2f} EUR")

        
    # Detailed variable output (only non-zero values)
    
    print("\n Non-zero decision variables")

    print("\n Trailer flow variables x_f")
    for (f, a), var in x_f.items():
        if var.X > 1e-6:
            print(f"x[{f}, {a}] = {var.X}")

    print("\n Driver flow variables y_d")
    for (d, a), var in y_d.items():
        if var.X > 1e-6:
            print(f"y[{d}, {a}] = {var.X}")

    print("\n Unserved shipments u")
    for f in F:
        if u[f].X > 1e-6:
            print(f"u[{f}] = {u[f].X}")


def plot_solution(m, vars_out, relays_df, home_bases_df, shipments_df,
                   title='Solution'):
    # Plot drivers and freight routing on a time-space diagram
    if m.Status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL, GRB.TIME_LIMIT):
        print('Skipp plot (no solution).')
        return

    F = vars_out['F']
    D = vars_out['D']
    I = vars_out['I']
    T = vars_out['T']
    x_f = vars_out['x_f']
    y_d = vars_out['y_d']
    u = vars_out['u']
    O_of = vars_out['O_of']
    D_of = vars_out['D_of']
    Ra = vars_out['Ra']
    Rd = vars_out['Rd']
    b_of = vars_out['b_of']

    # Plot physical location in y axis: home bases at top, then relays (arrival, departure), then origins, then destinations.
    y_pos = {}
    next_y = 0
    for d in D:
        if b_of[d] not in y_pos:
            y_pos[b_of[d]] = next_y
            next_y += 1
    next_y += 1
    for i in I:
        y_pos[Ra[i]] = next_y; next_y += 1
        y_pos[Rd[i]] = next_y; next_y += 1
    next_y += 1
    for f in F:
        y_pos[O_of[f]] = next_y; next_y += 1
        y_pos[D_of[f]] = next_y; next_y += 1

    fig, ax = plt.subplots(figsize=(14, 8))

    # Distribute all physical nodes across t
    for v, y in y_pos.items():
        ax.axhline(y, color='lightgray', linewidth=0.3, alpha=0.6)

    # Y-axis labels
    sorted_nodes = sorted(y_pos.items(), key=lambda kv: kv[1])
    ax.set_yticks([y for _, y in sorted_nodes])
    ax.set_yticklabels([v for v, _ in sorted_nodes], fontsize=7)
    ax.set_xlabel('Time period')
    ax.set_xlim(-1, T + 1)


    # Plot start time windows at origin, destination, RP as horizontal segments
    ship_df = shipments_df.set_index('shipment_id')
    relay_df = relays_df.set_index('relay_id')

    for f in F:
        # Origin time window
        t_start = int(ship_df.at[f, 't_O_under'])
        t_end   = int(ship_df.at[f, 't_O_bar'])
        y = y_pos[O_of[f]]

        ax.hlines(y=y, xmin=t_start, xmax=t_end,
              colors='green', linewidth=1, alpha=0.8,
              label='Origin start TW' if f == F[0] else None)

        # Destination time window
        t_start = int(ship_df.at[f, 't_D_under'])
        t_end   = int(ship_df.at[f, 't_D_bar'])
        y = y_pos[D_of[f]]

        ax.hlines(y=y, xmin=t_start, xmax=t_end,
              colors='purple', linewidth=1, alpha=0.8,
              label='Destination start TW' if f == F[0] else None)

    # Relays departure
    for i in I:
        t_start = int(relay_df.at[i, 't_id_under'])
        t_end   = int(relay_df.at[i, 't_id_bar'])
        y = y_pos[Rd[i]]

        ax.hlines(y=y, xmin=t_start, xmax=t_end,
              colors='orange', linewidth=1, alpha=0.8,
              label='Relay start TW' if i == I[0] else None)


    # Driver paths (one color per deployed driver)
    cmap = plt.cm.get_cmap('tab20')
    deployed = [d for d in D if any(y_d[(d, a)].X > 0.5
                                     for a in vars_out['A_acc'][d])]
    drv_colors = {d: cmap((i % 20) / 20.0) for i, d in enumerate(deployed)}

    seen_drv = set()
    for (d, a), var in y_d.items():
        if var.X <= 0.5:
            continue
        v1, v2, t1, t2 = a
        if v1 == v2:    # waiting at RP or Home base
            continue
        if v1 not in y_pos or v2 not in y_pos:
            continue
        col = drv_colors.get(d, 'gray')
        ax.plot([t1, t2], [y_pos[v1], y_pos[v2]],
                color=col, lw=1.6, alpha=0.85,
                label=f"Driver {d}" if d not in seen_drv else None)
        seen_drv.add(d)

    # Freight paths 
    parcel_label = False
    for (f, a), var in x_f.items():
        if var.X <= 0.5:
            continue
        v1, v2, t1, t2 = a
        if v1 == v2:
            continue
        if v1 not in y_pos or v2 not in y_pos:
            continue

        ax.plot([t1, t2], [y_pos[v1] + 0.2, y_pos[v2] + 0.2],
                color='blue', lw=1.2, linestyle='--', alpha=0.7,
                label='Freight path' if not parcel_label else None)
        parcel_label = True

    # Mark served & unserved shipments at their origin
    served = [f for f in F if u[f].X < 0.5]
    unserved = [f for f in F if u[f].X > 0.5]

    # Served shipments 
    if served:
        xs = [int(ship_df.at[f, 't_O_under']) for f in served]
        ys = [y_pos[O_of[f]] for f in served]
        ax.scatter(xs, ys, marker='x', color='blue', s=80,
                   label=f"Served ({len(served)})", zorder=7)

    # Unserved shipments 
    if unserved:
        xs = [int(ship_df.at[f, 't_O_under']) for f in unserved]
        ys = [y_pos[O_of[f]] for f in unserved]
        ax.scatter(xs, ys, marker='x', color='red', s=80,
                   label=f"Unserved ({len(unserved)})", zorder=7)

    ax.set_title(title)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.,
              fontsize=8)
    plt.subplots_adjust(right=0.78)
    plt.tight_layout()
    plt.show()


# MAIN


if __name__ == '__main__':
    INSTANCE_FILE = "G:/Documents/TUD/1. Thesis/data/data prun/instance_15.xlsx"

    params, relays_df, home_bases_df, shipments_df, distance_matrix_df = \
        load_instance(INSTANCE_FILE)

    print(f"\n Solving {INSTANCE_FILE}")
    m, vars_out = solve_relay_model(
        params, relays_df, home_bases_df, shipments_df, distance_matrix_df,
        time_limit_seconds=600,
        mip_gap=0.01,
        verbose=True,
    )

    print_solution(m, vars_out, instance_name=INSTANCE_FILE.split('/')[-1])
    plot_solution(m, vars_out, relays_df, home_bases_df, shipments_df,
                   title=f"Solution: {INSTANCE_FILE.split('/')[-1]}")